<a href="https://colab.research.google.com/github/anast752/-fintech-course-Maliukh/blob/main/practical6_crypto_volatility_Maliukh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests pandas plotly --quiet

In [2]:
import requests

import pandas as pd

coin_id = "ethereum"  # можна замінити на інший криптоактив, напр. "ethereum"

url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"

params = {"vs_currency": "usd", "days": "365", "interval": "daily"}

response = requests.get(url, params=params)

data = response.json()

prices_df = pd.DataFrame(data["prices"], columns=["timestamp", "price"])

prices_df["date"] = pd.to_datetime(prices_df["timestamp"], unit="ms")

prices_df = prices_df[["date", "price"]]

print(f"Завантажено {len(prices_df)} денних спостережень")

prices_df.head()

Завантажено 366 денних спостережень


,date,price
0,2025-10-26,3954.004120
1,2025-10-27,4158.567622
2,2025-10-28,4117.287078
3,2025-10-29,3978.812010
4,2025-10-30,3902.801947


In [3]:
import numpy as np

prices_df["log_return"] = np.log(prices_df["price"] / prices_df["price"].shift(1))

prices_df = prices_df.dropna()

daily_volatility = prices_df["log_return"].std()

annualized_volatility = daily_volatility * np.sqrt(365)

print(f"Щоденна волатильність: {daily_volatility:.4f}")

print(f"Річна (річна) волатильність: {annualized_volatility:.2%}")

Щоденна волатильність: 0.0319
Річна (річна) волатильність: 60.91%


In [4]:
!pip install yfinance --quiet

import yfinance as yf

benchmark = yf.download("^GSPC", period="1y", interval="1d")

benchmark["log_return"] = np.log(benchmark["Close"] / benchmark["Close"].shift(1))

benchmark = benchmark.dropna()

benchmark_daily_vol = benchmark["log_return"].std()

benchmark_annual_vol = benchmark_daily_vol * np.sqrt(252)  # 252 торгових дні на рік

print(f"Річна волатильність S&P 500: {benchmark_annual_vol:.2%}")

print(f"У скільки разів волатильність {coin_id} вища: {annualized_volatility / benchmark_annual_vol:.1f}x")

/tmp/ipykernel_3230/699541997.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  benchmark = yf.download("^GSPC", period="1y", interval="1d")
[*********************100%***********************]  1 of 1 completed

Річна волатильність S&P 500: 12.75%
У скільки разів волатильність ethereum вища: 4.8x


In [5]:
import plotly.graph_objects as go

prices_df["rolling_vol_30d"] = prices_df["log_return"].rolling(30).std() * np.sqrt(365)

fig = go.Figure()

fig.add_trace(go.Scatter(x=prices_df["date"], y=prices_df["price"], name="Ціна (USD)", yaxis="y1"))

fig.add_trace(go.Scatter(x=prices_df["date"], y=prices_df["rolling_vol_30d"], name="30-денна волатильність (річна)", yaxis="y2"))

fig.update_layout(

    title=f"Ціна та волатильність {coin_id}",

    yaxis=dict(title="Ціна, USD"),

    yaxis2=dict(title="Волатильність", overlaying="y", side="right"),

)

fig.show()